In [ ]:
import torch
import numpy as np

dtype = torch.float
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)


test 용 text

In [6]:
# input -파이에서 파이범위에서 데이터 생성 , 정답label은 sin(x)활용

x = np.linspace(-np.pi, np.pi, 1000)
y = np.sin(x)
x = torch.from_numpy(x)
y = torch.from_numpy(y)
print(x[1:20])

#가중치 a, b, c, d 준비
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)


tensor([-3.1353, -3.1290, -3.1227, -3.1164, -3.1101, -3.1039, -3.0976, -3.0913,
        -3.0850, -3.0787, -3.0724, -3.0661, -3.0598, -3.0535, -3.0473, -3.0410,
        -3.0347, -3.0284, -3.0221], dtype=torch.float64)


## Optimizer _ SGD 사용 , .step() , zero_grad() 를 통해 파라미터 업데이트

In [7]:
learning_rate = 1e-3

#예측값 return하는 함수
def compute_y_pred(x):
    return a + b * x + c * (x ** 2) + d * (x ** 3)

# 손실 함수: MSE
def compute_loss(y_true, y_pred):
    return torch.mean((y_pred - y_true) ** 2)

optim = torch.optim.SGD([a, b, c, d], lr=learning_rate)
crit = torch.nn.MSELoss(reduction='mean')
for t in range(2400):
    y_pred = compute_y_pred(x)
    loss = compute_loss(y, y_pred)
    if (t + 1) % 200 == 0:
        print(t + 1, loss.item())

    loss.backward()
    optim.step()
    optim.zero_grad()



print("a, b, c, d =", a.item(), b.item(), c.item(), d.item())

200 0.8109660129130577
400 0.5736335564837207
600 0.4062110572724537
800 0.28808059164886507
1000 0.2047143073208639
1200 0.1458705366294361
1400 0.1043287898570247
1600 0.07499694513187803
1800 0.054283226957115634
2000 0.039653485217305176
2200 0.02931940049880654
2400 0.022018710995233094
a, b, c, d = -0.1966371089220047 0.8301826119422913 0.03388781473040581 -0.089533731341362


In [8]:
criterion = torch.nn.MSELoss(reduction='mean')
optimizer = torch.optim.SGD([a, b, c, d], lr=1e-3)
for t in range(4000):
    y_pred = compute_y_pred(x)
    loss = criterion(y_pred, y)
    if (t + 1) % 200 == 0:
        print(t + 1, loss.item())
    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


200 0.01686042212579343
400 0.013215439479015993
600 0.0106395367638698
800 0.008818981948992272
1000 0.007532160941402411
1200 0.006622521882257247
1400 0.0059794585157599035
1600 0.005524816488819043
1800 0.005203363172934049
2000 0.004976067309868921
2200 0.004815338377293014
2400 0.004701675754650966
2600 0.0046212921242584585
2800 0.0045644410996149454
3000 0.004524231506714693
3200 0.004495791317232368
3400 0.004475674194424102
3600 0.004461444392764401
3800 0.004451378227708069
4000 0.004444257199420169


# **Image Classification**


In [9]:
from torchvision import datasets
from torchvision.transforms import ToTensor
training_data = datasets.FashionMNIST(
root="data", train=True, download=True, transform=ToTensor()
)
test_data = datasets.FashionMNIST(
root="data", train=False, download=True, transform=ToTensor()
)
img, label = training_data[0]
print(f'{label = }')
print(f'{type(img) = }')
print(f'{img.shape = }')
print()

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 271kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.02MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.14MB/s]


label = 9
type(img) = <class 'torch.Tensor'>
img.shape = torch.Size([1, 28, 28])



In [10]:
from torch.utils.data import DataLoader
batch_size = 64
new_batch_size = 32
train_dataloader = DataLoader(training_data, batch_size=new_batch_size ,shuffle=True) #배치단위로 작은데이터로 전체데이터를 학습하는 효과를 내기 위해  , 샘플링 데이터의 랜덤성 부여
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [11]:
device = (
"cuda"
if torch.cuda.is_available()
else "cpu"
)
print(f"Using {device} device")


Using cpu device


# **배치 사이즈에 맞게 데이터 적재**

In [12]:
import torch
import torch.nn as nn

m = nn.Flatten()
for X, y in train_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"{m(X).shape = }")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

import torch

# --------------------------------------------------------- 예시: train_dataloader에서 가져온 배치 데이터
for X, y in train_dataloader:
    # X를 28x28에서 784로 reshape
    X = X.reshape(-1, 784)

    # 출력
    print(f"Shape of X [N, 784]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")

    break  # 첫 번째 배치만 확인


Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
m(X).shape = torch.Size([64, 784])
Shape of y: torch.Size([64]) torch.int64
Shape of X [N, 784]: torch.Size([64, 784])
Shape of y: torch.Size([64]) torch.int64


# **모델 정의**

In [25]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(28*28, 512)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(512, 512)
        self.relu2 = nn.ReLU()
        self.linear3 = nn.Linear(512, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.linear1(x))
        x = self.relu2(self.linear2(x))
        return self.linear3(x)


# **nn.Sequential 사용**

In [24]:
class NeuralNetwork2(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )
    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)


# **Train-model**

In [15]:
def train(dataloader, model , loss_fn, optimizer):
  model.train()

  with torch.no_grad():
    for x,y in dataloader:
      pred = model(x)
      cnt=0
      loss = loss_fn(pred, y)
      if cnt % 4 ==0:

      loss.backward()

      optimizer.step()
      optimizer.zero_grad()


In [ ]:
def test(dataloader, model , loss_fn):
  model.eval()
  total_loss = 0
  correct = 0
  total = 0

  with torch.no_grad():
    for x,y in dataloader:
      pred = model(x)
      n += y.shape[0]
      y_pred = torch.argmax(pred, dim=1)
      c = (y ==pred)
      n_correct += c.sum().item()


  avg_loss = total_loss / len(dataloader)
  accuracy = n_correct / n
  print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")



In [43]:
def test(dataloader, model , loss_fn):
  model.eval()
  total_loss = 0
  correct = 0
  total = 0

  with torch.no_grad():
    for x,y in dataloader:
      pred = model(x)

      loss = loss_fn(pred, y)
      _, predicted = torch.max(pred, 1)
      correct += (predicted == y).sum().item()
      total += y.size(0)

  avg_loss = total_loss / len(dataloader)
  accuracy = correct / total * 100
  print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")



In [44]:
model = NeuralNetwork().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)



epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
    print("Done!")

Epoch 1
-------------------------------
Test Loss: 0.0000, Test Accuracy: 63.62%
Done!
Epoch 2
-------------------------------
Test Loss: 0.0000, Test Accuracy: 65.36%
Done!
Epoch 3
-------------------------------
Test Loss: 0.0000, Test Accuracy: 65.86%
Done!
Epoch 4
-------------------------------
Test Loss: 0.0000, Test Accuracy: 67.03%
Done!
Epoch 5
-------------------------------
Test Loss: 0.0000, Test Accuracy: 68.22%
Done!


In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. NeuralNetwork 모델 정의 (예시)
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)  # Flatten the input
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. 모델 정의
model = NeuralNetwork().to(device)

# 4. 손실 함수와 옵티마이저 정의
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

# 5. 학습 및 평가 함수 정의 (train, test는 미리 정의된 것으로 가정)
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

def test(dataloader, model, loss_fn):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total * 100
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

# 6. 학습 반복
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
    print("Done!")


Epoch 1
-------------------------------
Test Loss: 1.8077, Test Accuracy: 59.19%
Done!
Epoch 2
-------------------------------
Test Loss: 1.3938, Test Accuracy: 64.60%
Done!
Epoch 3
-------------------------------
Test Loss: 1.1556, Test Accuracy: 65.65%
Done!
Epoch 4
-------------------------------
Test Loss: 1.0197, Test Accuracy: 67.04%
Done!
Epoch 5
-------------------------------
Test Loss: 0.9342, Test Accuracy: 68.28%
Done!


배치사이즈를 1이나 2로 줄이면 학습이 불안정해져서 문제발생 - 왜?

배치사이즈